In [34]:
import struct
import numpy as np
import csv
import os
from scipy.integrate import simpson as simps
import matplotlib.pyplot as plt
from tqdm import trange



In [45]:
def csv_reader(file_name,save=False):
    data_csv = np.genfromtxt(file_name, delimiter=",", skip_header=3)
    length = len(data_csv[:, 0])
    size = length//1000
    print(size)
    time_csv = data_csv[:, 0]   # time column
    ch1_csv = data_csv[:, 1]    # channel 1
    ch2_csv = data_csv[:, 2]    # channel 2
    # ch3_csv = data_csv[:, 3]    # channel 3
    common_dtype = [
    ('event_number', np.int32),
    ('start_time', np.float64),
    ('baseline', np.float32),
    ('baseline_std', np.float32),
    ('1_peak_location', int),
    ('1_peak_value', np.float32),
    ('2_peak_location', int),
    ('2_peak_value', np.float32),
    ('1_peak_area', np.float32),
    ('2_peak_area', np.float32),]
    if save:
        common_dtype.append(('data', np.float32, 1000))
        nd_array = np.full(size, np.nan, dtype=common_dtype)
        channels_data ={1:nd_array.copy(),2:nd_array.copy(),3:nd_array.copy()}
        for i in trange (size,desc="Processing events"):
            time_i=time_csv[i*1000:(i+1)*1000]  ##2.5 ns time resolution    
            ch1_i=ch1_csv[i*1000:(i+1)*1000]*(-1)   
            ch2_i=ch2_csv[i*1000:(i+1)*1000]*(-1)    
            # ch3_i=ch3_csv[i*1000:(i+1)*1000]*(-1) 
            channels_data[1]['data'][i] = ch1_i
            channels_data[2]['data'][i] = ch2_i
            # channels_data[3]['data'][i] = ch3_i
            channels_data[1]['event_number'][i] = i
            channels_data[2]['event_number'][i] = i
            # channels_data[3]['event_number'][i] = i
            channels_data[1]['start_time'][i] = time_i[0]
            channels_data[2]['start_time'][i] = time_i[0]
            # channels_data[3]['start_time'][i] = time_i[0]
            channels_data[1]['baseline'][i] = np.average(ch1_i[0:30])
            channels_data[2]['baseline'][i] = np.average(ch2_i[0:30])
            # channels_data[3]['baseline'][i] = np.average(ch3_i[0:30])
            channels_data[1]['baseline_std'][i] = np.std(ch1_i[0:30])
            channels_data[2]['baseline_std'][i] = np.std(ch2_i[0:30])
            # channels_data[3]['baseline_std'][i] = np.std(ch3_i[0:30])
            channels_data[1]['1_peak_value'][i] = np.max(ch1_i)
            channels_data[2]['1_peak_value'][i] = np.max(ch2_i)
            # channels_data[3]['1_peak_value'][i] = np.max(ch3_i)
            channels_data[1]['1_peak_location'][i] = int(np.argmax(ch1_i))
            channels_data[2]['1_peak_location'][i] = int(np.argmax(ch2_i))
            # channels_data[3]['1_peak_location'][i] = int(np.argmax(ch3_i))
            ## 33 samples is 338(2.5*(2000/781)) ns sepeartion from peak side, have to be to the right for muon decay
            
            ind_ch1_peak_1_l = int(np.argmax(ch1_i))-33
            ind_ch2_peak_1_l = int(np.argmax(ch2_i))-33
            # ind_ch3_peak_1_l = int(np.argmax(ch3_i))-33
            if ind_ch1_peak_1_l < 0:
                ind_ch1_peak_1_l = 0
            if ind_ch2_peak_1_l < 0:
                ind_ch2_peak_1_l = 0
            # if ind_ch3_peak_1_l < 0:
            #     ind_ch3_peak_1_l = 0
            ind_ch1_peak_1_r = int(np.argmax(ch1_i))+33
            ind_ch2_peak_1_r = int(np.argmax(ch2_i))+33
            # ind_ch3_peak_1_r = int(np.argmax(ch3_i))+33
            if ind_ch1_peak_1_r > 999:
                ind_ch1_peak_1_r = 999
            if ind_ch2_peak_1_r > 999:
                ind_ch2_peak_1_r = 999
            # if ind_ch3_peak_1_r > 999:
            #     ind_ch3_peak_1_r = 999
            
            channels_data[1]['2_peak_value'][i] = np.max(ch1_i[ind_ch1_peak_1_r:])
            channels_data[2]['2_peak_value'][i] = np.max(ch2_i[ind_ch2_peak_1_r:])
            # channels_data[3]['2_peak_value'][i] = np.max(ch3_i[ind_ch3_peak_1_r:])
            channels_data[1]['2_peak_location'][i] = int(
                np.argmax(ch1_i[ind_ch1_peak_1_r:])+ind_ch1_peak_1_r)
            channels_data[2]['2_peak_location'][i] = int(
                np.argmax(ch2_i[ind_ch2_peak_1_r:])+ind_ch2_peak_1_r)
            # channels_data[3]['2_peak_location'][i] = int(
            #     np.argmax(ch3_i[ind_ch3_peak_1_r:])+ind_ch3_peak_1_r)
            channels_data[1]['1_peak_area'][i] =  simps(ch1_i[ind_ch1_peak_1_l:ind_ch1_peak_1_r],
                                                              time_i[ind_ch1_peak_1_l:ind_ch1_peak_1_r])
            channels_data[2]['1_peak_area'][i] =  simps(ch2_i[ind_ch2_peak_1_l:ind_ch2_peak_1_r],
                                                              time_i[ind_ch2_peak_1_l:ind_ch2_peak_1_r])
            # channels_data[3]['1_peak_area'][i] =  simps(ch3_i[ind_ch3_peak_1_l:ind_ch3_peak_1_r],
            #                                                   time_i[ind_ch3_peak_1_l:ind_ch3_peak_1_r]) 
            ind_ch1_peak_2_l = channels_data[1]['2_peak_location'][i]-33
            ind_ch2_peak_2_l = channels_data[2]['2_peak_location'][i]-33
            # ind_ch3_peak_2_l = channels_data[3]['2_peak_location'][i]-33
            if ind_ch1_peak_2_l < 0:
                ind_ch1_peak_2_l = 0
            if ind_ch2_peak_2_l < 0:
                ind_ch2_peak_2_l = 0
            # if ind_ch3_peak_2_l < 0:
            #     ind_ch3_peak_2_l = 0
            ind_ch1_peak_2_r = channels_data[1]['2_peak_location'][i]+33
            ind_ch2_peak_2_r = channels_data[2]['2_peak_location'][i]+33
            # ind_ch3_peak_2_r = channels_data[3]['2_peak_location'][i]+33
            if ind_ch1_peak_2_r > 999:
                ind_ch1_peak_2_r = 999
            if ind_ch2_peak_2_r > 999:
                ind_ch2_peak_2_r = 999
            # if ind_ch3_peak_2_r > 999:
            #     ind_ch3_peak_2_r = 999           
            channels_data[1]['2_peak_area'][i] =  simps(
                ch1_i[ind_ch1_peak_2_l:ind_ch1_peak_2_r],
                    time_i[ind_ch1_peak_2_l:ind_ch1_peak_2_r])
            channels_data[2]['2_peak_area'][i] =  simps(
                ch2_i[ind_ch2_peak_2_l:ind_ch2_peak_2_r],
                    time_i[ind_ch2_peak_2_l:ind_ch2_peak_2_r])
            # channels_data[3]['2_peak_area'][i] =  simps(
            #     ch3_i[ind_ch3_peak_2_l:ind_ch3_peak_2_r],
            #         time_i[ind_ch3_peak_2_l:ind_ch3_peak_2_r])

            
        return(channels_data)

    else:    
        nd_array = np.full(size, np.nan, dtype=common_dtype)
        channels_data ={1:nd_array.copy(),2:nd_array.copy(),3:nd_array.copy()}
        for i in trange (size,desc="Processing events"):
            time_i=time_csv[i*1000:(i+1)*1000] ##2.5*(2000/781) ns time resolution    
            ch1_i=ch1_csv[i*1000:(i+1)*1000]*(-1)     
            ch2_i=ch2_csv[i*1000:(i+1)*1000]*(-1)     
            # ch3_i=ch3_csv[i*1000:(i+1)*1000]*(-1) 
            channels_data[1]['event_number'][i] = i
            channels_data[2]['event_number'][i] = i
            # channels_data[3]['event_number'][i] = i
            channels_data[1]['start_time'][i] = time_i[0]
            channels_data[2]['start_time'][i] = time_i[0]
            # channels_data[3]['start_time'][i] = time_i[0]
            channels_data[1]['baseline'][i] = np.average(ch1_i[0:30])
            channels_data[2]['baseline'][i] = np.average(ch2_i[0:30])
            # channels_data[3]['baseline'][i] = np.average(ch3_i[0:30])
            channels_data[1]['baseline_std'][i] = np.std(ch1_i[0:30])
            channels_data[2]['baseline_std'][i] = np.std(ch2_i[0:30])
            # channels_data[3]['baseline_std'][i] = np.std(ch3_i[0:30])
            channels_data[1]['1_peak_value'][i] = np.max(ch1_i)
            channels_data[2]['1_peak_value'][i] = np.max(ch2_i)
            # channels_data[3]['1_peak_value'][i] = np.max(ch3_i)
            channels_data[1]['1_peak_location'][i] = int(np.argmax(ch1_i))
            channels_data[2]['1_peak_location'][i] = int(np.argmax(ch2_i))
            # channels_data[3]['1_peak_location'][i] = int(np.argmax(ch3_i))
            ## 33 samples is 338(2.5*(2000/781)) ns sepeartion from peak side, have to be to the right for muon decay
            
            ind_ch1_peak_1_l = int(np.argmax(ch1_i))-33
            ind_ch2_peak_1_l = int(np.argmax(ch2_i))-33
            # ind_ch3_peak_1_l = int(np.argmax(ch3_i))-33
            if ind_ch1_peak_1_l < 0:
                ind_ch1_peak_1_l = 0
            if ind_ch2_peak_1_l < 0:
                ind_ch2_peak_1_l = 0
            # if ind_ch3_peak_1_l < 0:
            #     ind_ch3_peak_1_l = 0
            ind_ch1_peak_1_r = int(np.argmax(ch1_i))+33
            ind_ch2_peak_1_r = int(np.argmax(ch2_i))+33
            # ind_ch3_peak_1_r = int(np.argmax(ch3_i))+33
            if ind_ch1_peak_1_r > 999:
                ind_ch1_peak_1_r = 999
            if ind_ch2_peak_1_r > 999:
                ind_ch2_peak_1_r = 999
            # if ind_ch3_peak_1_r > 999:
            #     ind_ch3_peak_1_r = 999
            
            channels_data[1]['2_peak_value'][i] = np.max(ch1_i[ind_ch1_peak_1_r:])
            channels_data[2]['2_peak_value'][i] = np.max(ch2_i[ind_ch2_peak_1_r:])
            # channels_data[3]['2_peak_value'][i] = np.max(ch3_i[ind_ch3_peak_1_r:])
            channels_data[1]['2_peak_location'][i] = int(
                np.argmax(ch1_i[ind_ch1_peak_1_r:])+ind_ch1_peak_1_r)
            channels_data[2]['2_peak_location'][i] = int(
                np.argmax(ch2_i[ind_ch2_peak_1_r:])+ind_ch2_peak_1_r)
            # channels_data[3]['2_peak_location'][i] = int(
            #     np.argmax(ch3_i[ind_ch3_peak_1_r:])+ind_ch3_peak_1_r)
            channels_data[1]['1_peak_area'][i] =  simps(ch1_i[ind_ch1_peak_1_l:ind_ch1_peak_1_r],
                                                              time_i[ind_ch1_peak_1_l:ind_ch1_peak_1_r])
            channels_data[2]['1_peak_area'][i] =  simps(ch2_i[ind_ch2_peak_1_l:ind_ch2_peak_1_r],
                                                              time_i[ind_ch2_peak_1_l:ind_ch2_peak_1_r])
            # channels_data[3]['1_peak_area'][i] =  simps(ch3_i[ind_ch3_peak_1_l:ind_ch3_peak_1_r],
            #                                                   time_i[ind_ch3_peak_1_l:ind_ch3_peak_1_r]) 
            ind_ch1_peak_2_l = channels_data[1]['2_peak_location'][i]-33
            ind_ch2_peak_2_l = channels_data[2]['2_peak_location'][i]-33
            # ind_ch3_peak_2_l = channels_data[3]['2_peak_location'][i]-33
            if ind_ch1_peak_2_l < 0:
                ind_ch1_peak_2_l = 0
            if ind_ch2_peak_2_l < 0:
                ind_ch2_peak_2_l = 0
            # if ind_ch3_peak_2_l < 0:
            #     ind_ch3_peak_2_l = 0
            ind_ch1_peak_2_r = channels_data[1]['2_peak_location'][i]+33
            ind_ch2_peak_2_r = channels_data[2]['2_peak_location'][i]+33
            # ind_ch3_peak_2_r = channels_data[3]['2_peak_location'][i]+33
            if ind_ch1_peak_2_r > 999:
                ind_ch1_peak_2_r = 999
            if ind_ch2_peak_2_r > 999:
                ind_ch2_peak_2_r = 999
            # if ind_ch3_peak_2_r > 999:
            #     ind_ch3_peak_2_r = 780           
            channels_data[1]['2_peak_area'][i] =  simps(
                ch1_i[ind_ch1_peak_2_l:ind_ch1_peak_2_r],
                    time_i[ind_ch1_peak_2_l:ind_ch1_peak_2_r])
            channels_data[2]['2_peak_area'][i] =  simps(
                ch2_i[ind_ch2_peak_2_l:ind_ch2_peak_2_r],
                    time_i[ind_ch2_peak_2_l:ind_ch2_peak_2_r])
            # channels_data[3]['2_peak_area'][i] =  simps(
            #     ch3_i[ind_ch3_peak_2_l:ind_ch3_peak_2_r],
            #         time_i[ind_ch3_peak_2_l:ind_ch3_peak_2_r])

        return(channels_data)
    
    


In [46]:
procceseced_data=csv_reader("/Users/drorta/alpha-1/measurements/nd-2-measurement-0.csv",save=True)

/Users/drorta/jupyter-env/lib/python3.14/site-packages/numpy/_core/numeric.py:386: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(a, fill_value, casting='unsafe')


640


Processing events: 100%|██████████| 640/640 [00:00<00:00, 3086.39it/s]


In [47]:
ch_1,ch_2=procceseced_data[1],procceseced_data[2]

In [48]:
# for i in range (0,1):
#     if i == 1:
#         procceseced_data=csv_reader(f"/Users/drorta/alpha-1/measurements/nd-2-measurement-{i}.csv",save=True)
#         ch_1,ch_2=procceseced_data[1],procceseced_data[2]
#     else:
#         procceseced_data=csv_reader(f"/Users/drorta/alpha-1/measurements/nd-2-measurement-{i}.csv",save=True)
#         ch_1_t,ch_2_t=procceseced_data[1],procceseced_data[2]
#         ch_1_t['event_number'] = ch_1_t['event_number']+ch_1['event_number'][-1]
#         ch_1_t['start_time'] = ch_1_t['start_time']+ch_1['start_time'][-1]
#         ch_2_t['event_number'] = ch_2_t['event_number']+ch_2['event_number'][-1]
#         ch_1 = np.append(ch_1,ch_1_t)
#         ch_2 = np.append(ch_2,ch_2_t)
#         del ch_1_t , ch_2_t

In [50]:
len(ch_1['event_number'])
# np.unique((ch_1['event_number']),return_counts=True)

640